Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [7]:
import sys, json, warnings
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_DIR = PROJECT_ROOT / "conf"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="train")

experiment_config = cfg.experiment
hyperparams = cfg.hyperparams
segment_frames = hyperparams["segment_frames"]
batch_size = hyperparams["batch_size"]

warnings.filterwarnings("ignore", message=".*nested tensors.*")

In [8]:
# NOTE: dataset import path is from the old notebook; adjust it if the dataset moved into `tracers`.
from tracers.datasets import AutoOsu4KDataset, split_song_ids, make_dataloader

ROOT = PROJECT_ROOT / "dataset"
AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"

# The base dataset is only used to reproduce the same song split as training.
base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    audio_cache_size=8,
)
train_ids, val_ids, _ = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)

train_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    crops_per_chart=2,
    song_ids=train_ids,
)

val_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=val_ids,
)

val_loader = make_dataloader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
train_loader = make_dataloader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [9]:
from tracers.models.transformer import TracerNpModel
from tracers.training.trainer import Trainer
from tracers.generate.converter import predictions_to_json, export_generation_debug

model = TracerNpModel(hyperparams).to(device)

# The optimizer is only here because load_checkpoint() also restores its state.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"],
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=None,
    valid_loader=None,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

CHECKPOINT = PROJECT_ROOT / "notebooks/checkpoints/best.pt"
trainer.load_checkpoint(CHECKPOINT)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 76, best val loss 0.1494)


In [12]:
# for fname in ["best.pt", "latest.pt"]:
#     trainer.evaluate(val_loader, checkpoint=ckpt_dir / fname, name=fname)

trainer.evaluate(train_loader, checkpoint=CHECKPOINT, name="TRAIN")

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 76, best val loss 0.1494)


Validation: 100%|██████████| 180/180 [00:47<00:00,  3.76it/s]


TRAIN (48s)
Loss   | Total: 0.1210 | NP: 0.0096 | NS: 0.0539
Onset  | P: 0.8475 | R: 0.8090 | F1: 0.8278
Action | P: 0.8987 | R: 0.8988 | F1: 0.8987
Active Exact Accuracy: 0.4329
GT Active Rate: 4.99% | Predicted Active Rate: 4.99%


{'losses': {'total': 0.1210176279490403,
  'np': 0.009581993102461083,
  'ns': 0.05394367621444116},
 'metrics': {'onset_precision': 0.8475097847644344,
  'onset_recall': 0.8089977996939676,
  'onset_f1': 0.8278061115158787,
  'action_precision': 0.8987106393196375,
  'action_recall': 0.898785922509019,
  'action_f1': 0.8987482793378138,
  'active_exact_accuracy': 0.4328906660114148,
  'gt_active_rate': 0.04985133630289532,
  'predicted_active_rate': 0.04985551224944321}}